# Study D, Milestone 2
Decision: **GO**, scoped to a methods-and-re-analysis paper. See `reports/milestone2_decision.md`.

In [1]:
import json, sys, pandas as pd, numpy as np
sys.path.insert(0, '..')
pd.set_option('display.width', 220)

## Synthetic ground truth: mechanism recovery (E9, primary tumour)
`naive` is the variance of clone-level state fractions, the statistic a clone analysis reports by default.

In [2]:
d = pd.read_csv('../results/tables/e9_mechanism_recovery.csv')
d[d.organ == 0].groupby('mech').agg(truth_B=('truth_B','median'), B=('B','median'),
    naive=('clone_frac_var','median'), frac_CI_excludes_0=('B_lo', lambda s: (s>0).mean()),
    reps=('B','size')).round(4)

,truth_B,B,naive,frac_CI_excludes_0,reps
mech,,,,,
A_stable,0.0000,0.0000,0.0403,0.0,15
B_switching,0.0779,0.0605,0.0684,0.0,15
C_induction,0.0000,0.0000,0.0024,0.0,15
D_bottleneck,0.0000,0.0000,0.0023,0.0,15
E_coupled,0.0665,0.0549,0.0664,0.4,15


### Covariance curves by MRCA depth (the signature)

In [3]:
c = pd.read_csv('../results/tables/e9_curves.csv')
c[c.organ==0].pivot_table(index='depth', columns='mech', values='site_centred').round(4)

mech,A_stable,B_switching,C_induction,D_bottleneck,E_coupled
depth,,,,,
0,0.0027,0.0656,-0.0000,-0.0005,0.0300
1,0.0186,0.0766,-0.0003,0.0007,0.0317
2,0.0071,0.0623,0.0064,-0.0027,0.0315
3,0.0511,0.0902,0.0143,-0.0017,0.0258
4,0.0342,0.0751,0.0139,0.0092,0.0429
5,0.0253,0.0632,0.0471,-0.0047,0.0500
6,0.1152,0.0679,-0.0054,0.0381,0.0574
7,0.0823,0.0752,NaN,0.0595,0.0538
8,0.0541,NaN,NaN,NaN,NaN


### Stress tests: misclassification and clone-label loss attenuate B, never inflate it

In [4]:
s = pd.read_csv('../results/tables/e9_stress.csv'); s = s[s.organ==0]
s.groupby(['eps','clone_loss','n_sample']).B.median().round(4).unstack()

n_sample           800     4000
eps  clone_loss                
0.00 0.0         0.0300  0.0453
     0.3         0.0000  0.0172
0.05 0.0         0.0574  0.0463
     0.3         0.0522  0.0336
0.15 0.0         0.0141  0.0193
     0.3         0.0107  0.0280

## Real data: Simeonov GSE173958
### Where the metastatic EMT enrichment comes from

In [5]:
pd.read_csv('../results/tables/m2_real_c3_composition.csv').round(3)

,mouse,site,total,composition,within_clone
0,1,Met,0.399,0.471,-0.072
1,1,Liver,0.163,0.397,-0.234
2,1,Lung,0.302,0.467,-0.165
3,2,Met,-0.049,0.037,-0.087
4,2,Liver,-0.002,0.159,-0.161
5,2,Lung,-0.344,-0.132,-0.212


### Technical control: many unrelated programmes shift as much as EMT

In [6]:
t = pd.read_csv('../results/tables/m2_real_c1_technical_control.csv')
t.pivot_table(index='feature', columns=['mouse','site'], values='within_clone_shift_sd').round(2)

mouse            1                 2            
site         Liver  Lung   Met Liver  Lung   Met
feature                                         
adhesion      0.03 -0.06 -0.20  0.04  0.13 -0.09
chromatin    -0.41 -0.55 -0.24 -0.18 -0.20 -0.37
e            -0.31 -0.23 -0.13 -0.15 -0.12 -0.09
glyco        -0.20 -0.85 -0.36 -0.67 -1.48 -0.56
hypoxia      -0.35 -0.86 -0.55 -0.56 -0.83 -0.38
ifn           0.06 -0.03 -0.04 -0.09  0.01 -0.08
nFeature_RNA  0.23  0.57  0.00 -0.75 -1.09 -1.05
percent_mt    0.06 -0.01 -0.15 -0.21 -0.09 -0.07
percent_ribo  0.47  0.28  0.71  1.02  0.60  0.91
prolif        0.42  0.61  0.34  0.24 -0.72  0.10
s100          0.53  0.87  0.38  0.42 -0.16  0.37
stem         -0.13 -0.27 -0.23  0.31  0.59 -0.05
stress       -0.38 -0.41 -0.28 -0.04 -0.64 -0.37

### State heritability (clone-stratified permutation)

In [7]:
pd.read_csv('../results/tables/m2_real_c2_h2_permutation.csv').round(4)

,mouse,site,n_pairs,obs_spearman,null_mean,null_sd,p_perm
0,1,PT,127432,-0.0444,0.0157,0.0150,0.00
1,1,Met,375401,-0.0915,-0.0011,0.0129,0.00
2,1,Liver,313096,-0.0569,-0.0010,0.0126,0.00
3,2,PT,23540,-0.0321,-0.0413,0.0147,0.71
4,2,Met,372820,-0.0006,-0.0015,0.0171,0.52
5,2,Liver,341264,0.0341,0.0000,0.0170,0.98


### The strict test: heritable switching propensity

In [8]:
pd.read_csv('../results/tables/m2_real_decomposition.csv')[
  ['mouse','site','n_clones','n_cells','k','A','B','B_lo','B_hi']].round(4)

,mouse,site,n_clones,n_cells,k,A,B,B_lo,B_hi
0,1,PT,9,2494,0.0200,0.1206,0.0000,0.0000,0.0435
1,1,Met,5,3691,0.0200,0.1085,0.0000,0.0000,0.0342
2,1,Liver,4,2724,0.0200,0.0000,0.1077,0.0000,0.1692
3,1,PTab,3,1149,0.0200,0.0032,0.0000,0.0000,0.0000
4,1,Blood,2,408,0.0200,0.0037,0.0000,0.0000,0.0000
5,2,PT,19,1619,0.3296,0.0000,0.0417,0.0045,0.0810
6,2,Met,5,8760,0.0200,0.0367,0.0000,0.0000,0.0668
7,2,Liver,5,3369,0.0200,0.0000,0.0461,0.0000,0.1280


### Model comparison and held-out site prediction

In [9]:
pd.read_csv('../results/tables/m2_real_model_comparison.csv')[
  ['mouse','model','npar','loglik','aic','delta_aic']].round(2)

,mouse,model,npar,loglik,aic,delta_aic
0,1,M0,2,-4294.83,8593.67,250.46
1,1,M1,8,-4163.60,8343.20,0.00
2,1,M2,3,-4293.54,8593.07,249.87
3,1,M4,9,-4178.05,8374.10,30.90
4,2,M0,2,-8895.67,17795.33,439.87
5,2,M1,20,-8677.88,17395.76,40.29
6,2,M2,3,-8902.82,17811.65,456.18
7,2,M4,21,-8656.73,17355.46,0.00


In [10]:
h = pd.read_csv('../results/tables/m2_real_heldout_site.csv')
h.groupby('mouse')[['err_M0','err_M1']].mean().round(4)

,err_M0,err_M1
mouse,,
1,0.2884,0.1824
2,0.2085,0.0885


In [11]:
json.load(open('../results/metrics/m2_real_metrics.json'))

{'M1_cells_with_tree': 10608,
 'M1_clones': 29,
 'M1_pt_quartile_cuts': [-0.5617974565489168,
  0.04825502077483695,
  0.5231429815899339],
 'M1_emt_vs_authors_pseudotime_spearman': [0.7550719682391374, 0.0],
 'M1_H1_clone_PTfracLH_vs_aggression_spearman': [-0.05347407899331232,
  0.8384861477956523,
  17],
 'M2_cells_with_tree': 13834,
 'M2_clones': 66,
 'M2_pt_quartile_cuts': [-0.41871560621286197,
  0.036249357926695966,
  0.45338220564968024],
 'M2_H1_clone_PTfracLH_vs_aggression_spearman': [0.14798713353289727,
  0.4189177824194725,
  32]}